## Week 6 Day 3: Context Engineering with MCP (LangChain 버전)

원본 노트북의 OpenAI Agents SDK 코드를 LangChain / LangGraph 에이전트로 바꾼 것입니다.

바뀐 부분은 세 군데뿐입니다.

1. `MCPServerStdio` -> `MultiServerMCPClient` (`langchain-mcp-adapters`)
2. `Agent` + `Runner.run` -> `create_agent` + `agent.ainvoke`
3. `create_static_tool_filter` -> 파이썬 리스트 필터링, `trace(...)` -> LangSmith 자동 추적

필요한 패키지:

```
pip install langchain langchain-mcp-adapters langchain-openai python-dotenv
```

# 세팅

In [1]:
from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient  # MCP 서버에 붙어서 도구를 가져오는 클라이언트
from langchain.agents import create_agent                        # LangChain v1 의 에이전트 생성 함수
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv(override=True)

True

### 윈도우 사용자를 위한 참고

노트북에서 로컬 MCP 서버(stdio)를 띄울 때 윈도우에서만 걸리는 문제가 있습니다. 서버는 시작 로그를 stderr 로 쓰는데, 윈도우 주피터 커널의 stderr 스트림에는 실제 파일 핸들이 없어서 `io.UnsupportedOperation: fileno` 로 실패합니다. 맥과 리눅스는 해당 없습니다.

해결책은 서버의 stderr 를 널 디바이스로 보내는 것입니다. 원본 노트북은 `agents.mcp.server.stdio_client` 를 패치했지만, LangChain 에서는 `langchain_mcp_adapters.sessions` 안의 `stdio_client` 를 같은 방식으로 패치합니다.

`langchain-mcp-adapters` 0.3.2 기준으로 `StdioConnection` 에는 `errlog` 옵션이 없어서, 아래처럼 패치하는 방법 외에 설정으로 끄는 방법은 확인되지 않았습니다.

In [3]:
import sys

if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

Applied the Windows adjustment


## Part 2: 웹 검색

모델은 학습한 내용까지만 압니다. 웹 검색은 모델을 최신 상태로 유지해 주는 컨텍스트 소스입니다.

Tavily 는 에이전트용으로 만들어진 검색 API 이고, 자체 MCP 서버를 제공합니다.

무료 API 키가 필요합니다.

1. https://www.tavily.com 에서 가입
2. 무료 플랜은 월 1,000회 검색, 카드 등록 불필요
3. `tvly-` 로 시작하는 키를 `.env` 에 추가

`TAVILY_API_KEY=tvly-xxxx`

In [4]:
tavily_client = MultiServerMCPClient({
    "tavily": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "tavily-mcp@latest"],
        "env": {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")},
    }
})

tavily_tools = await tavily_client.get_tools()
tavily_tools

[StructuredTool(name='tavily_search', description='Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.', args_schema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}, 'search_depth': {'type': 'string', 'enum': ['basic', 'advanced', 'fast', 'ultra-fast'], 'description': "The depth of the search. 'basic' for generic results, 'advanced' for more thorough search, 'fast' for optimized low latency with high relevance, 'ultra-fast' for prioritizing latency above all else", 'default': 'basic'}, 'topic': {'type': 'string', 'enum': ['general'], 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'default': 'general'}, 'time_range': {'type': 'string', 'description': 'The time range back from the current date to include in the search results', 'enum': ['day', 'week', 'month', 'year']}, 'start_date': {

Tavily 서버는 search, extract, crawl, map, research 등 여러 도구를 제공합니다. 여기서는 일반 웹 검색만 쓰고 싶으므로 `tavily_search` 만 남깁니다.

원본은 OpenAI Agents SDK 의 `create_static_tool_filter` 를 썼지만, LangChain 에서는 도구가 그냥 파이썬 리스트이므로 리스트를 걸러 주면 끝입니다. 에이전트에게 어떤 도구를 줄지 고르는 것 자체가 컨텍스트 엔지니어링입니다.

In [5]:

# 이름이 tavily_search 인 도구만 남깁니다.
search_only = [tool for tool in tavily_tools if tool.name == "tavily_search"]


In [6]:
# 도구를 사용하는 에이전트를 생성하는 함수
from langchain.agents import create_agent
# 에이전트의 역할을 정의하는 시스템 지침 (한글)
instructions = "당신은 웹을 검색해 정보를 찾고 핵심 내용을 간결하게 요약합니다."

# 에이전트에게 전달할 실제 요청문 (한글)
# f-string으로 현재 날짜를 문자열에 삽입 → 모델이 "최신"의 기준 시점을 알 수 있게 함
request = f"아마존(Amazon) 주가에 대한 최신 뉴스를 조사하고 향후 전망을 간략히 요약해 주세요. 참고로 오늘 날짜는 {datetime.now().strftime('%Y-%m-%d')}입니다."


# 모델 + 도구 + 시스템 지침으로 에이전트를 만든다
agent = create_agent("openai:gpt-5-mini", search_only , system_prompt=instructions)


In [7]:

# messages 키에 사용자 입력을 넣어 비동기로 실행한다
result = await agent.ainvoke({"messages": request})

# 마지막 메시지가 에이전트의 최종 답변이다
print(result["messages"][-1].content)

요약 (2026-09-09 시점)

핵심 뉴스
- 실적·클라우드 모멘텀: 최근 분기에서 AWS 성장세가 가속화됐고(분기별 보도에 따르면 분기 성장률이 20%대~37% 수준으로 보고됨), 광고·소매 부문도 견조한 매출을 기록해 총수익이 예상치를 상회한 보도 다수(예: Yahoo Finance, Barchart, 247wallst).  
- 비경상 이익 영향: 과거 분기(특히 2026년 상반기)에 Anthropic 지분 평가이익 등으로 대규모 비영업성 이익(수십억 달러 규모)이 반영돼 순이익이 급증했다는 점이 지적됨(Barchart, FinanceFeeds).  
- 대규모 투자(캡엑스) 증가: 회사는 AI·데이터센터·칩 등 인프라 투자 확대를 발표했고, 관련 보도에서는 연간 캡엑스 전망이 종전 수치에서 크게 늘어나 $125B → $200B+ (일부 보도에선 $220B) 범위로 언급됨(Outlook Business, Yahoo Finance, 기타).  
- 인력 구조조정: 수익성 개선을 위한 감원 진행(회사 발표 기준 수천~만 단위의 감원 보도)이 계속되고 있음(여러 매체).  
- 시장 반응·애널리스트: 실적 직후 주가가 급등한 사례(두 자릿수 상승)와, 캡엑스 확대·현금흐름 우려로 급락한 케이스가 병존. 애널리스트들은 대체로 AWS 중심의 중장기 낙관 전망을 유지하면서도(목표주가 범위 약 $275~$343 등), 단기 리스크(현금흐름·투자 규모)를 경고함(Economic Times, Finance Yahoo, Barchart).

주가(시장) 영향 요약
- 긍정 요인: AWS의 강한 성장, 광고·리테일의 회복, Anthropic 등 AI 관련 자산 가치 상승 가능성 — 장기 성장 스토리(특히 AI 인프라) 지지.  
- 부정 요인: 막대한 캡엑스(현금유출)로 인한 자유현금흐름(FCF) 압박, 일회성 평가이익에 의한 실적 왜곡 가능성, 클라우드 경쟁 심화(MS/GOOGL), 실행 리스크(대규모 투자·조직 재편).

향후 전망(간단)
- 단기(분기~1년): 변동

## 정리

MCP 서버 네 개, 컨텍스트 네 종류: 장기 기억, 웹 검색, 에이전트가 스스로 만든 지식 베이스, 살아 있는 외부 연동.

### OpenAI Agents SDK 와의 대응표

| 원본 | LangChain |
| --- | --- |
| `MCPServerStdio(params=...)` | `MultiServerMCPClient({"이름": {...}})` |
| `await server.list_tools()` | `await client.get_tools()` |
| `Agent(name=, instructions=, model=, mcp_servers=)` | `create_agent(model, tools, system_prompt=)` |
| `await Runner.run(agent, request)` | `await agent.ainvoke({"messages": [...]})` |
| `result.final_output` | `result["messages"][-1].content` |
| `create_static_tool_filter(allowed_tool_names=[...])` | 파이썬 리스트 필터링 |
| `max_turns=20` | `{"recursion_limit": 40}` |
| `with trace(...)` | LangSmith 환경변수 |

### 알아 둘 차이 하나

`MultiServerMCPClient` 는 `async with` 로 서버를 붙잡아 두지 않고, 도구가 호출될 때마다 새 세션(= 새 서버 프로세스)을 엽니다. 그래서 코드가 짧아지는 대신 호출마다 프로세스 기동 비용이 붙습니다. 세션을 유지하려면 `client.session("이름")` 컨텍스트 매니저와 `langchain_mcp_adapters.tools.load_mcp_tools(session)` 을 쓰면 됩니다.